In [2]:
import pandas as pd
import numpy as np
from sklearn import datasets, ensemble
from sklearn.metrics import explained_variance_score, max_error

from evidently import ColumnMapping
from evidently.base_metric import InputData
from evidently.metrics import RegressionQualityMetric
from evidently.metrics.custom_metric import CustomValueMetric
from evidently.report import Report
from evidently.renderers.html_widgets import WidgetSize
from evidently.test_suite import TestSuite
from evidently.tests.custom_test import CustomValueTest

# Configuration des colonnes
TARGET_COLUMN = "target"
PREDICTION_COLUMN = "prediction"

# Charger le dataset de cancer du sein
bcancer_data = datasets.load_breast_cancer(as_frame=True)
bcancer_df = bcancer_data.frame
feature_names = bcancer_data.feature_names.tolist()

# Créer les échantillons de référence et actuel
bcancer_ref = bcancer_df.sample(n=300, random_state=42)
bcancer_cur = bcancer_df.drop(bcancer_ref.index).sample(n=269, random_state=42)

# Entraîner le modèle de régression (on utilise target comme variable continue)
regressor = ensemble.RandomForestRegressor(random_state=0, n_estimators=50)
regressor.fit(bcancer_ref[feature_names], bcancer_ref[TARGET_COLUMN])

# Faire les prédictions
bcancer_ref[PREDICTION_COLUMN] = regressor.predict(bcancer_ref[feature_names])
bcancer_cur[PREDICTION_COLUMN] = regressor.predict(bcancer_cur[feature_names])

# Configuration du mapping des colonnes
column_mapping = ColumnMapping()
column_mapping.target = TARGET_COLUMN
column_mapping.prediction = PREDICTION_COLUMN
column_mapping.numerical_features = feature_names

# === MÉTRIQUES PERSONNALISÉES ===

def explained_variance_func(data: InputData):
    """
    Calcule le Score de Variance Expliquée (Explained Variance Score)
    
    Le score de variance expliquée mesure la proportion de variance 
    dans les données cibles qui est expliquée par le modèle.
    Valeur proche de 1.0 = excellent modèle
    """
    return explained_variance_score(
        data.current_data[data.column_mapping.target],
        data.current_data[data.column_mapping.prediction]
    )

def max_error_func(data: InputData):
    """
    Calcule l'Erreur Maximale (Max Error)
    
    L'erreur maximale est la plus grande erreur absolue entre 
    les valeurs prédites et les valeurs réelles.
    Plus faible = meilleur modèle
    """
    return max_error(
        data.current_data[data.column_mapping.target],
        data.current_data[data.column_mapping.prediction]
    )

def comparison_explained_variance_func(data: InputData):
    """
    Compare le Score de Variance Expliquée entre référence et données actuelles
    Valeur positive = amélioration, négative = dégradation
    """
    if data.reference_data is not None:
        ref_score = explained_variance_score(
            data.reference_data[data.column_mapping.target],
            data.reference_data[data.column_mapping.prediction]
        )
        cur_score = explained_variance_score(
            data.current_data[data.column_mapping.target],
            data.current_data[data.column_mapping.prediction]
        )
        return cur_score - ref_score  # Différence entre actuel et référence
    else:
        return explained_variance_score(
            data.current_data[data.column_mapping.target],
            data.current_data[data.column_mapping.prediction]
        )

def comparison_max_error_func(data: InputData):
    """
    Compare l'Erreur Maximale entre référence et données actuelles
    Valeur négative = amélioration, positive = dégradation
    """
    if data.reference_data is not None:
        ref_error = max_error(
            data.reference_data[data.column_mapping.target],
            data.reference_data[data.column_mapping.prediction]
        )
        cur_error = max_error(
            data.current_data[data.column_mapping.target],
            data.current_data[data.column_mapping.prediction]
        )
        return cur_error - ref_error  # Différence entre actuel et référence
    else:
        return max_error(
            data.current_data[data.column_mapping.target],
            data.current_data[data.column_mapping.prediction]
        )

def model_performance_ratio_func(data: InputData):
    """
    Métrique personnalisée : ratio de performance
    Calcule le ratio entre variance expliquée et erreur maximale normalisée
    """
    variance_score = explained_variance_score(
        data.current_data[data.column_mapping.target],
        data.current_data[data.column_mapping.prediction]
    )
    max_err = max_error(
        data.current_data[data.column_mapping.target],
        data.current_data[data.column_mapping.prediction]
    )
    
    # Normaliser l'erreur maximale par la plage des valeurs cibles
    target_range = data.current_data[data.column_mapping.target].max() - data.current_data[data.column_mapping.target].min()
    normalized_max_error = max_err / target_range if target_range > 0 else 0
    
    # Retourner un ratio (plus élevé = meilleur)
    return variance_score / (normalized_max_error + 0.001)  # +0.001 pour éviter division par zéro

# === CRÉATION DU RAPPORT ===

print("=== RAPPORT DE MÉTRIQUES PERSONNALISÉES ===")
print(f"Dataset: Cancer du Sein")
print(f"Données de référence: {len(bcancer_ref)} échantillons")
print(f"Données actuelles: {len(bcancer_cur)} échantillons")
print(f"Features: {len(feature_names)} caractéristiques")
print()

# Rapport avec les métriques personnalisées
custom_metrics_report = Report(
    metrics=[
        RegressionQualityMetric(),  # Métrique standard de qualité de régression
        
        # Métriques personnalisées - valeurs actuelles
        CustomValueMetric(
            func=explained_variance_func, 
            title="Current: Explained Variance Score", 
            size=WidgetSize.HALF
        ),
        CustomValueMetric(
            func=max_error_func, 
            title="Current: Max Error", 
            size=WidgetSize.HALF
        ),
        
        # Métriques personnalisées - comparaisons
        CustomValueMetric(
            func=comparison_explained_variance_func, 
            title="Diff: Explained Variance Score (Current - Reference)", 
            size=WidgetSize.HALF
        ),
        CustomValueMetric(
            func=comparison_max_error_func, 
            title="Diff: Max Error (Current - Reference)", 
            size=WidgetSize.HALF
        ),
        
        # Métrique personnalisée avancée
        CustomValueMetric(
            func=model_performance_ratio_func, 
            title="Custom: Performance Ratio (Variance/Normalized_Error)", 
            size=WidgetSize.FULL
        ),
    ]
)

# Exécuter le rapport
custom_metrics_report.run(
    reference_data=bcancer_ref,
    current_data=bcancer_cur,
    column_mapping=column_mapping,
)

# Afficher le rapport
custom_metrics_report.show(mode='inline')

# === EXEMPLE D'UTILISATION AVEC TEST SUITE ===

print("\n=== TESTS PERSONNALISÉS ===")

# Test suite avec des seuils adaptés au dataset de cancer du sein
custom_test_suite = TestSuite(tests=[
    # Test pour le score de variance expliquée 
    # (pour données binaires en régression, on s'attend à des valeurs élevées)
    CustomValueTest(
        func=explained_variance_func, 
        title='Explained Variance Score Test', 
        lte=1.0,  # Moins ou égal à 1.0
        gte=0.8   # Plus ou égal à 0.8 (seuil élevé car données binaires)
    ),
    
    # Test pour l'erreur maximale 
    # (pour des valeurs binaires 0/1, erreur max devrait être proche de 1)
    CustomValueTest(
        func=max_error_func, 
        title='Max Error Test', 
        lte=1.0   # L'erreur maximale ne doit pas dépasser 1.0
    ),
    
    # Test pour le ratio de performance personnalisé
    CustomValueTest(
        func=model_performance_ratio_func, 
        title='Performance Ratio Test', 
        gte=10.0  # Le ratio doit être au moins 10 pour être considéré comme bon
    )
])

# Exécuter la suite de tests
custom_test_suite.run(
    reference_data=bcancer_ref,
    current_data=bcancer_cur,
    column_mapping=column_mapping
)

# Afficher les résultats des tests
custom_test_suite.show(mode='inline')

# === AFFICHAGE DES VALEURS CALCULÉES ===

print("\n=== VALEURS DES MÉTRIQUES PERSONNALISÉES ===")

# Calculer manuellement les métriques pour affichage
from evidently.base_metric import InputData

# Créer un objet InputData pour les calculs
input_data = InputData(
    reference_data=bcancer_ref,
    current_data=bcancer_cur,
    column_mapping=column_mapping,
    data_definition=None,
    reference_additional_features=None,
    current_additional_features=None,
    additional_data={}
)

explained_var = explained_variance_func(input_data)
max_err = max_error_func(input_data)
performance_ratio = model_performance_ratio_func(input_data)

print(f"📊 Explained Variance Score: {explained_var:.4f}")
print(f"❌ Max Error: {max_err:.4f}")
print(f"⚡ Performance Ratio: {performance_ratio:.2f}")

if explained_var > 0.9:
    print("✅ Excellent modèle - Variance expliquée très élevée")
elif explained_var > 0.8:
    print("✅ Bon modèle - Variance expliquée élevée")
else:
    print("⚠️  Modèle à améliorer - Variance expliquée faible")

print(f"\n💡 Note: Pour un dataset binaire (0/1), une erreur maximale de {max_err:.4f} est {'acceptable' if max_err <= 1.0 else 'élevée'}")

=== RAPPORT DE MÉTRIQUES PERSONNALISÉES ===
Dataset: Cancer du Sein
Données de référence: 300 échantillons
Données actuelles: 269 échantillons
Features: 30 caractéristiques


=== TESTS PERSONNALISÉS ===

=== VALEURS DES MÉTRIQUES PERSONNALISÉES ===
📊 Explained Variance Score: 0.8015
❌ Max Error: 1.0000
⚡ Performance Ratio: 0.80
✅ Bon modèle - Variance expliquée élevée

💡 Note: Pour un dataset binaire (0/1), une erreur maximale de 1.0000 est acceptable
